In [1]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
df_org=pd.read_csv('Telco_churn_dataset.csv')

In [3]:
df_org.sample(4)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
6633,4415-IJZTP,Female,0,No,No,1,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,74.50,74.5,Yes
6170,2220-IAHLS,Female,0,No,No,1,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Credit card (automatic),19.40,19.4,No
6608,4576-CSAJH,Male,0,No,No,22,Yes,No,DSL,No,...,No,Yes,No,No,One year,Yes,Credit card (automatic),55.15,1193.05,Yes
1981,0348-SDKOL,Female,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,Yes,Yes,Yes,Yes,Month-to-month,Yes,Credit card (automatic),102.10,5885.4,Yes


In [4]:
X = df_org.drop('Churn', axis=1)
Y = df_org['Churn']

In [5]:
Y = Y.map({
    'Yes':1,
    'No':0
})
Y = Y.astype(int)

In [6]:
Y.head()

0    0
1    0
2    1
3    0
4    1
Name: Churn, dtype: int32

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,Y,test_size=0.2,random_state=42)

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (FunctionTransformer)
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Evaluation Metrics
from sklearn.metrics import (

    accuracy_score,

    classification_report,

    confusion_matrix
)




num_columns = [
    'tenure',
    'MonthlyCharges',
    'TotalCharges'
]

multi_columns = [
    'MultipleLines',
    'InternetService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
    'Contract',
    'PaymentMethod'
]

binary_columns = [
    'gender',
    'Partner',
    'Dependents',
    'PhoneService',
    'PaperlessBilling',
]


def basic_cleaning(df1):

    df1 = df1.copy()

    # Remove customerID
    df1 = df1.drop('customerID', axis=1)

    # Convert TotalCharges
    df1['TotalCharges'] = pd.to_numeric(df1['TotalCharges'],errors='coerce')

    # Fill missing values
    df1['TotalCharges'] = df1['TotalCharges'].fillna(df1['TotalCharges'].median())

    # Binary mapping
    binary_map = {
        'Yes':1,
        'No':0,
        'Female':0,
        'Male':1
    }

    for col in binary_columns:
        df1[col] = df1[col].map(binary_map)

    return df1

In [11]:
preprocessor = ColumnTransformer([('num', StandardScaler(), num_columns ),
                                  ('cat', OneHotEncoder(drop='first'),multi_columns)
                                  ],remainder='passthrough')

In [17]:
results = []

# Different values
eps_values = [0.3, 0.5, 0.8, 1.0, 1.5, 2.0]
min_samples_values = [3, 5, 7, 10]

for eps in eps_values:
    for min_samp in min_samples_values:

        # DBSCAN model
        model = DBSCAN(
            eps=eps,
            min_samples=min_samp
        )

        # Pipeline
        dbscan_pipe = Pipeline([
            ('cleaning', FunctionTransformer(basic_cleaning)),
            ('preprocessing', preprocessor)
        ])

        # Transform data
        X_transformed = dbscan_pipe.fit_transform(X_train)

        # Fit DBSCAN
        clusters = model.fit_predict(X_transformed)

        # Number of clusters
        n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)

        # Number of noise points
        noise_points = list(clusters).count(-1)

        # Silhouette Score
        if n_clusters > 1:
            score = silhouette_score(X_transformed, clusters)
        else:
            score = -1

        # Store results
        results.append({
            'eps': eps,
            'min_samples': min_samp,
            'clusters_found': n_clusters,
            'noise_points': noise_points,
            'silhouette_score': score
        })

# Convert to DataFrame
results_df = pd.DataFrame(results)

# Sort by best silhouette score
results_df = results_df.sort_values(
    by='silhouette_score',
    ascending=False
)

print(results_df)

    eps  min_samples  clusters_found  noise_points  silhouette_score
20  2.0            3               2             5          0.145159
21  2.0            5               2             6          0.141670
22  2.0            7               2            11          0.137056
23  2.0           10               2            20          0.135844
19  1.5           10               5          1448          0.051850
18  1.5            7               8          1189          0.012801
17  1.5            5              18           917         -0.043135
16  1.5            3              36           550         -0.109495
6   0.5            7              50          4937         -0.176804
8   0.8            3             262          4015         -0.187178
11  0.8           10              27          5091         -0.193529
2   0.3            7              42          5106         -0.196992
7   0.5           10              23          5204         -0.205939
4   0.5            3             2